In [1]:
# =====================================================================
# CELL 1: SETUP & MOUNT SHARED DRIVE
# =====================================================================
from google.colab import drive
import os

drive.mount('/content/drive')

# IMPORTANT: If your shared folder shortcut has a different name, change it here!
SHARED_FOLDER_PATH = "/content/drive/MyDrive/GSV_Math_Model_Cache"

STAR_CHECKPOINT_DIR = f"{SHARED_FOLDER_PATH}/star_finetuned_qwen"
VDS_RESULTS_FILE = f"{SHARED_FOLDER_PATH}/gsv_math_results/qwen2.5_star_BLIND.json"

os.makedirs(STAR_CHECKPOINT_DIR, exist_ok=True)
os.makedirs(os.path.dirname(VDS_RESULTS_FILE), exist_ok=True)

print("Drive mounted and folders ready!")

Mounted at /content/drive
Drive mounted and folders ready!


In [2]:
# =====================================================================
# CELL 2: INSTALL UNSLOTH (For ultra-fast, memory-efficient fine-tuning)
# =====================================================================
!pip install unsloth
!pip install --force-reinstall "xformers<0.0.28" # Fix for specific Colab CUDA issues
!pip install bitsandbytes accelerate datasets tqdm qwen-vl-utils

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.3/76.3 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.4/86.4 MB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 44.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 122.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 36.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 87.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 117.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 124.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3

In [3]:
# =====================================================================
# CELL 3: LOAD QWEN 2.5-VL (UNSLOTH OPTIMIZED)
# =====================================================================
from unsloth import FastVisionModel
import torch

MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"

print(f"Loading {MODEL_ID} for STaR Fine-tuning...")
model, tokenizer = FastVisionModel.from_pretrained(
    MODEL_ID,
    load_in_4bit=True, # Essential for Colab Free Tier (T4 16GB)
    use_gradient_checkpointing="unsloth",
)
print("Model loaded successfully!")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Loading Qwen/Qwen2.5-VL-7B-Instruct for STaR Fine-tuning...
==((====))==  Unsloth 2026.8.19: Fast Qwen2_5_Vl patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

Model loaded successfully!


In [19]:
# =====================================================================
# CELL 4: LOAD & FORMAT TRAINING DATASET (Safe Collator Format)
# =====================================================================
from datasets import load_dataset, concatenate_datasets, get_dataset_config_names

print("Fetching available math datasets...")
all_configs = get_dataset_config_names("lmms-lab/LLaVA-OneVision-Data")
math_configs = [c for c in all_configs if ('math' in c.lower() or 'geo' in c.lower() or 'science' in c.lower()) and 'clevr' not in c.lower()]

datasets_list = []
for config in math_configs[:3]:
    print(f"Downloading {config}...")
    datasets_list.append(load_dataset("lmms-lab/LLaVA-OneVision-Data", config, split="train"))

dataset = concatenate_datasets(datasets_list).shuffle(seed=42).select(range(5000))

def format_clean(example):
    try:
        convo = example["conversations"]
        user_text = convo[0]["value"].replace("<image>", "").strip()
        assistant_text = convo[1]["value"]

        # 💥 Perfect JSON format: We do NOT embed the PIL Image inside the dictionary!
        messages = [
            {"role": "user", "content": [{"type": "image"}, {"type": "text", "text": user_text}]},
            {"role": "assistant", "content": [{"type": "text", "text": assistant_text}]}
        ]

        # We put the raw PIL image in its own dedicated column for the collator
        return {"messages": messages, "images": [example["image"]]}
    except Exception:
        return {"messages": [], "images": []}

print("Formatting dataset...")
train_dataset = dataset.map(format_clean, remove_columns=dataset.column_names)
train_dataset = train_dataset.filter(lambda x: len(x["messages"]) > 0)

print(f"Prepared {len(train_dataset)} STaR training examples.")

Fetching available math datasets...
Formatting dataset...


Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/5000 [00:00<?, ? examples/s]

Prepared 5000 STaR training examples.


In [10]:
# =====================================================================
# CELL 5: INJECT LORA ADAPTERS (Only trains ~2% of the model)
# =====================================================================
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=False, # Keep vision encoder frozen to preserve visual grounding
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    random_state=3407,
)
print("LoRA adapters injected! Ready for training.")

RuntimeError: Unsloth: You already added LoRA adapters to your model!

In [21]:
# =====================================================================
# CELL 6: SETUP TRAINER (Fixed HuggingFace Syntax)
# =====================================================================
from unsloth import UnslothVisionDataCollator
from transformers import Trainer, TrainingArguments
import torch

training_args = TrainingArguments(
    output_dir=STAR_CHECKPOINT_DIR,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=50,
    max_steps=500,
    learning_rate=2e-4,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=10,
    save_strategy="steps",
    save_steps=50,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="linear",
    seed=3407,
    remove_unused_columns=False, # CRITICAL: Allows the collator to see the 'images' column
)

trainer = Trainer(
    model=model,
    processing_class=tokenizer, # 💥 FIX: HuggingFace renamed 'tokenizer' to 'processing_class'
    train_dataset=train_dataset,
    data_collator=UnslothVisionDataCollator(model, tokenizer),
    args=training_args,
)

Unsloth: Model does not have a default image size - using 512


In [22]:
# =====================================================================
# CELL 7: START STaR FINE-TUNING
# =====================================================================
print("Starting STaR fine-tuning! Checkpoints will stream to your Shared Drive.")

# If you get disconnected, change resume_from_checkpoint=True
trainer.train(resume_from_checkpoint=False)

# Save the absolute final weights
final_save_path = f"{STAR_CHECKPOINT_DIR}/final_model"
model.save_pretrained(final_save_path)
tokenizer.save_pretrained(final_save_path)

print(f" Training Complete! Final model saved to {final_save_path}")

Starting STaR fine-tuning! Checkpoints will stream to your Shared Drive.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 5,000 | Num Epochs = 1 | Total steps = 500
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 40,370,176 of 8,332,536,832 (0.48% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
10,7.836260
20,3.686528
30,1.688732
40,0.649425
50,0.460934
60,0.406210
70,0.362499
80,0.360685
90,0.413153
100,0.413100


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/GSV_Math_Model_Cache/star_finetuned_qwen/checkpoint-50/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/GSV_Math_Model_Cache/star_finetuned_qwen/checkpoint-100/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/GSV_Math_Model_Cache/star_finetuned_qwen/checkpoint-150/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/GSV_Math_Model_Cache/star_finetuned_qwen/checkpoint-200/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/GSV_Math_Model_Cache/star_finetuned_qwen/checkpoint-250/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/GSV_Math_Model_Cache/star_finetuned_qwen/checkpoint-300/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/GSV_Math_

 Training Complete! Final model saved to /content/drive/MyDrive/GSV_Math_Model_Cache/star_finetuned_qwen/final_model


In [4]:
# =====================================================================
# CELL 9: SMART VDS AGGREGATION & METRICS (MathVista TestMini)
# =====================================================================
import json
import re
import os
import glob
from datasets import load_dataset
from collections import defaultdict

print("Hunting for your VDS results file...")

# 1. Automatically find the most recent VDS results file
search_paths = [
    "/content/*.json",                                              # Local Colab storage
    "/content/drive/MyDrive/GSV_Math_Model_Cache/**/*.json",        # Your specific project folder
    "/content/drive/MyDrive/*.json"                                 # Root of Drive
]

found_files = []
for path in search_paths:
    found_files.extend(glob.glob(path, recursive=True))

vds_files = [f for f in found_files if "vds" in f.lower() or "results" in f.lower() or "blind" in f.lower()]

if not vds_files:
    raise FileNotFoundError("Could not find your JSON file! Make sure Google Drive is mounted (if it was saved there).")

vds_files.sort(key=os.path.getmtime, reverse=True)
VDS_RESULTS_FILE = vds_files[0]
print(f"✅ Automatically found and using: {VDS_RESULTS_FILE}")

# 2. Load MathVista
print("\nLoading MathVista metadata to match categories...")
dataset = load_dataset("AI4Math/MathVista", split="testmini")
metadata_map = {str(item["pid"]): item for item in dataset}

with open(VDS_RESULTS_FILE, "r") as f:
    results = json.load(f)

print(f"Loaded {len(results)} evaluated samples.")

# 3. Smart Parsing Extraction Logic (Now regex-safe!)
def smart_extract(response, ground_truth, q_type):
    response = str(response).strip()
    gt = str(ground_truth).strip()

    if "The answer is" in response:
        ans_part = response.split("The answer is")[-1]
        if gt.lower() in ans_part.lower():
            return True

    if q_type == "multi_choice":
        # Escape the ground truth so + or * don't break the regex
        escaped_gt = re.escape(gt)
        # Use custom lookarounds instead of \b to handle math symbols safely
        match = re.search(rf"(?<![a-zA-Z0-9]){escaped_gt}(?![a-zA-Z0-9])", response, re.IGNORECASE)
        if match:
            return True

    gt_nums = re.findall(r'-?\d*\.?\d+', gt)
    pred_nums = re.findall(r'-?\d*\.?\d+', response)

    if gt_nums and pred_nums:
        if gt_nums[-1] == pred_nums[-1]:
            return True
        if gt_nums[0] in pred_nums:
            return True

    if gt.lower() in response.lower():
        return True

    return False

# 4. Categorized Grading Setup
categories = {
    "Question Format": defaultdict(lambda: {"correct": 0, "total": 0}),
    "Answer Type": defaultdict(lambda: {"correct": 0, "total": 0}),
    "Mathematical Task": defaultdict(lambda: {"correct": 0, "total": 0}),
    "Grade Level": defaultdict(lambda: {"correct": 0, "total": 0}),
}

total_correct = 0
total_samples = 0

# 5. Grade the outputs
for pred_data in results:
    pid = str(pred_data["pid"])
    response = pred_data.get("raw_response", "")

    meta = metadata_map.get(pid)
    if not meta:
        continue

    total_samples += 1
    gt = meta["answer"]
    q_format = meta["question_type"]
    ans_type = meta["answer_type"]

    meta_dict = json.loads(meta["metadata"]) if isinstance(meta["metadata"], str) else meta["metadata"]
    math_task = meta_dict.get("task", "Unknown")
    grade_level = meta_dict.get("grade", "Unknown")

    is_correct = smart_extract(response, gt, q_format)

    if is_correct:
        total_correct += 1

    for cat_name, cat_val in [
        ("Question Format", q_format),
        ("Answer Type", ans_type),
        ("Mathematical Task", math_task),
        ("Grade Level", grade_level)
    ]:
        categories[cat_name][cat_val]["total"] += 1
        if is_correct:
            categories[cat_name][cat_val]["correct"] += 1

# 6. Print Markdown Tables
print(f"\n### MathVista Performance Breakdown (Smart Graded - FINETUNED BLIND)")
print(f"\n**OVERALL ACCURACY:** {total_correct/total_samples*100:.2f}% ({total_correct} / {total_samples})\n")

for cat_name, cat_data in categories.items():
    print(f"| **{cat_name}** | **Accuracy** | **Correct / Total** |")
    print("|---|---|---|")
    for sub_cat, stats in sorted(cat_data.items()):
        acc = stats["correct"] / stats["total"] * 100 if stats["total"] > 0 else 0
        print(f"| {sub_cat} | {acc:.2f}% | {stats['correct']} / {stats['total']} |")
    print("\n")

Hunting for your VDS results file...
✅ Automatically found and using: /content/drive/MyDrive/GSV_Math_Model_Cache/gsv_math_results/qwen2.5_star_BLIND.json

Loading MathVista metadata to match categories...
Loaded 1000 evaluated samples.

### MathVista Performance Breakdown (Smart Graded - FINETUNED BLIND)

**OVERALL ACCURACY:** 6.90% (69 / 1000)

| **Question Format** | **Accuracy** | **Correct / Total** |
|---|---|---|
| free_form | 13.04% | 60 / 460 |
| multi_choice | 1.67% | 9 / 540 |


| **Answer Type** | **Accuracy** | **Correct / Total** |
|---|---|---|
| float | 0.00% | 0 / 40 |
| integer | 14.11% | 59 / 418 |
| list | 50.00% | 1 / 2 |
| text | 1.67% | 9 / 540 |


| **Mathematical Task** | **Accuracy** | **Correct / Total** |
|---|---|---|
| figure question answering | 11.52% | 31 / 269 |
| geometry problem solving | 0.00% | 0 / 208 |
| math word problem | 12.37% | 23 / 186 |
| textbook question answering | 3.80% | 6 / 158 |
| visual question answering | 5.03% | 9 / 179 |


| **

In [25]:
# =====================================================================
# CELL 9: FULL METRICS AGGREGATION & SMART GRADING
# =====================================================================
import json, os, re
from datasets import load_dataset
from collections import defaultdict

# ---------------------------------------------------------
# 1. THE SMART PARSER (To grade the raw responses correctly)
# ---------------------------------------------------------
FINAL_ANSWER_PATTERNS = [
    r'\\boxed\{([^}]*)\}',
    r'[Ff]inal\s*[Aa]nswer\s*[:\-]?\s*(.{1,80})',
    r'[Tt]herefore[,\\s]+(?:the\s+)?(?:answer|value|result)\s+is\s*[:\-]?\s*(.{1,80})',
    r'[Tt]he\s+answer\s+is\s*[:\-]?\s*(.{1,80})',
    r'[Ss]o\s+the\s+answer\s+is\s*[:\-]?\s*(.{1,80})',
    r'=\s*(\S+)\s*$',
]

def extract_final_answer_region(raw_text, tail_chars=300):
    for pattern in FINAL_ANSWER_PATTERNS:
        matches = list(re.finditer(pattern, raw_text, re.IGNORECASE | re.DOTALL))
        if matches: return matches[-1].group(1).strip()
    return raw_text[-tail_chars:] if len(raw_text) > tail_chars else raw_text

def clean_free_form(text):
    if not isinstance(text, str): return str(text)
    text = text.strip().lower()
    for prefix in ["the answer is", "therefore, the answer is", "so the answer is", "the value is", "answer is", "value is", "equals", "it is", "the final answer is", "final answer:", "answer:"]:
        if text.startswith(prefix): text = text[len(prefix):].strip()
    match = re.match(r'^[a-zA-Z\s]+=\s*(.*)$', text)
    if match: text = match.group(1).strip()
    return text.rstrip('.!?*, ')

def get_most_similar(extraction, choices):
    if not choices: return extraction
    distances = [-len(set(extraction.lower()) & set(choice.lower())) for choice in choices]
    return choices[distances.index(min(distances))]

def normalize_extracted_answer(extraction, choices, question_type, answer_type):
    extraction = str(extraction).strip() if extraction else ""
    extraction = extract_final_answer_region(extraction)

    if question_type == 'multi_choice':
        letter = re.findall(r'\(([a-zA-Z])\)', extraction)
        extraction = letter[0].upper() if letter else extraction
        options = [chr(ord('A') + i) for i in range(len(choices))]
        if extraction in options:
            extraction = choices[options.index(extraction)]
        else:
            extraction = get_most_similar(clean_free_form(extraction), choices)
    else:
        cleaned = clean_free_form(extraction)
        if answer_type in ['integer', 'float']:
            numbers = re.findall(r'-?\d+\.?\d*', cleaned)
            extraction = numbers[-1] if numbers else cleaned
        else:
            extraction = cleaned
    return extraction

def is_correct(pred, gt, answer_type):
    if str(pred).lower().strip() == str(gt).lower().strip(): return 1
    if answer_type in ['integer', 'float']:
        try:
            if abs(float(pred) - float(gt)) < 1e-5: return 1
        except: pass
    return 0


# ---------------------------------------------------------
# 2. LOAD DATA & GRADE ON THE FLY
# ---------------------------------------------------------
RESULTS_FILE = "/content/drive/MyDrive/GSV_Math_Model_Cache/gsv_math_results/qwen2.5_star_BLIND.json"

if not os.path.exists(RESULTS_FILE):
    print(f"File not found: {RESULTS_FILE}")
else:
    print("Loading MathVista metadata to match categories...")
    dataset = load_dataset("AI4Math/MathVista", split="testmini")
    metadata_lookup = {sample["pid"]: sample for sample in dataset}

    with open(RESULTS_FILE, "r") as f:
        results = json.load(f)

    print(f"Loaded {len(results)} evaluated samples.\n")

    overall = {"correct": 0, "total": 0}
    by_q_type = defaultdict(lambda: {"correct": 0, "total": 0})
    by_a_type = defaultdict(lambda: {"correct": 0, "total": 0})
    by_task = defaultdict(lambda: {"correct": 0, "total": 0})
    by_grade = defaultdict(lambda: {"correct": 0, "total": 0})

    for res in results:
        pid = res["pid"]
        raw_response = str(res.get("raw_response", ""))
        gt = str(res["ground_truth"])

        # Get metadata
        meta = metadata_lookup.get(pid, {})
        q_type = meta.get("question_type", "Unknown")
        a_type = meta.get("answer_type", "Unknown")
        choices = meta.get("choices", [])

        # MathVista hides deep metadata inside a nested string/dict
        deep_meta = meta.get("metadata", {})
        if isinstance(deep_meta, str):
            try: deep_meta = json.loads(deep_meta)
            except: deep_meta = {}

        task = deep_meta.get("task", "Unknown")
        grade = deep_meta.get("grade", "Unknown")

        #  RE-GRADE IT USING THE SMART PARSER
        parsed_ans = normalize_extracted_answer(raw_response, choices, q_type, a_type)
        correct = is_correct(parsed_ans, gt, a_type)

        # Tally up
        overall["correct"] += correct
        overall["total"] += 1

        by_q_type[q_type]["correct"] += correct; by_q_type[q_type]["total"] += 1
        by_a_type[a_type]["correct"] += correct; by_a_type[a_type]["total"] += 1
        by_task[task]["correct"] += correct; by_task[task]["total"] += 1
        by_grade[grade]["correct"] += correct; by_grade[grade]["total"] += 1

    def print_table_section(title, data_dict):
        print(f"| **{title}** | **Accuracy** | **Correct / Total** |")
        print("|---|---|---|")
        for key in sorted(data_dict.keys()):
            c = data_dict[key]["correct"]
            t = data_dict[key]["total"]
            pct = (c / t) * 100 if t > 0 else 0
            print(f"| {key} | {pct:.2f}% | {c} / {t} |")
        print("\n")

    print("### MathVista Performance Breakdown (Smart Graded)\n")

    c, t = overall['correct'], overall['total']
    pct = (c / t) * 100 if t > 0 else 0
    print(f"**OVERALL ACCURACY:** {pct:.2f}% ({c} / {t})\n")

    print_table_section("Question Format", by_q_type)
    print_table_section("Answer Type", by_a_type)
    print_table_section("Mathematical Task", by_task)
    print_table_section("Grade Level", by_grade)

Loading MathVista metadata to match categories...
Loaded 1000 evaluated samples.

### MathVista Performance Breakdown (Smart Graded)

**OVERALL ACCURACY:** 32.60% (326 / 1000)

| **Question Format** | **Accuracy** | **Correct / Total** |
|---|---|---|
| free_form | 9.57% | 44 / 460 |
| multi_choice | 52.22% | 282 / 540 |


| **Answer Type** | **Accuracy** | **Correct / Total** |
|---|---|---|
| float | 0.00% | 0 / 40 |
| integer | 10.53% | 44 / 418 |
| list | 0.00% | 0 / 2 |
| text | 52.22% | 282 / 540 |


| **Mathematical Task** | **Accuracy** | **Correct / Total** |
|---|---|---|
| figure question answering | 23.05% | 62 / 269 |
| geometry problem solving | 56.25% | 117 / 208 |
| math word problem | 13.44% | 25 / 186 |
| textbook question answering | 44.94% | 71 / 158 |
| visual question answering | 28.49% | 51 / 179 |


| **Grade Level** | **Accuracy** | **Correct / Total** |
|---|---|---|
| college | 26.79% | 30 / 112 |
| daily life | 25.20% | 96 / 381 |
| elementary school | 14.43